# A product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

In [4]:
pip install selenium webdriver-manager beautifulsoup4

  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB 1.4 MB/s eta 0:00:07
   ---------------------------------------- 0.1/9.6 MB 1.3 MB/s eta 0:00:08
    --------------------------------------- 0.2/9.6 MB 1.6 MB/s eta 0:00:07
   - -------------------------------------- 0.3/9.6 MB 1.8 MB/s eta 0:00:06
   - -------------------------------------- 0.4/9.6 MB 1.9 MB/s eta 0:00:05
   -- ------------------------------------- 0.5/9.6 MB 2.0 MB/s eta 0:00:05
   -- ------------------------------------- 0.6/9.6 MB 2.1 MB/s eta 0:00:05
   --- ------------------------------------ 0.7/9.6 MB 2.1 MB/s eta 0:00:05
   --- ------------------------------------ 0.8/9.6 MB 2.2 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/9.6 MB 2.2 MB/s eta 0:00:05
   ---- ----------------------------------- 1.1/9.6 MB 2.2 MB/s eta 0:00:04
   ---- -----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kubernetes 34.1.0 requires urllib3<2.4.0,>=1.24.2, but you have urllib3 2.5.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [5]:

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup 

class Website:

    def __init__(self, url):
        self.url = url
        self.title = "No title found"
        self.links = []
        self.text = ""

        options = webdriver.ChromeOptions()
        options.add_argument("--headless")  
        options.add_argument("start-maximized")
        options.add_argument("--disable-gpu")
        
        options.add_experimental_option('excludeSwitches', ['enable-logging'])
        
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), 
            options=options
        )

        try:
            driver.get(url)
            
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            self.title = driver.title
            
            try:
                link_elements = driver.find_elements(By.TAG_NAME, 'a')
                links = [
                    link.get_attribute('href') 
                    for link in link_elements
                ]
                self.links = [link for link in links if link]
            except NoSuchElementException:
                self.links = []

            html_content = driver.page_source
            
            soup = BeautifulSoup(html_content, 'html.parser')
            
            irrelevant_tags = ["script", "style", "img", "input", "footer", "header", "nav", "button", "aside"]
            
            for tag in irrelevant_tags:
                for element in soup.find_all(tag):
                    element.decompose()

            extracted_parts = []
            for element in soup.find_all(['h1', 'h2', 'h3', 'p'], limit=500):
                text = element.get_text(strip=True)
                if not text:
                    continue
                    
                if element.name == 'h1':
                    extracted_parts.append(f"\n# {text}\n")
                elif element.name == 'h2':
                    extracted_parts.append(f"\n## {text}\n")
                elif element.name == 'h3':
                    extracted_parts.append(f"\n### {text}\n")
                elif element.name == 'p':
                    extracted_parts.append(f"{text}\n")

            self.text = "\n".join(extracted_parts).strip()
            
        except TimeoutException:
            self.title = f"Timeout Error on {url}"
        except Exception as e:
            self.title = f"Scraping Error"
        finally:
            driver.quit()

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\n\nWebpage Contents (Structured):\n{self.text}\n\n"

In [12]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/#wp--skip-link--target',
 'https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com/',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwardd

In [13]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example. You will be given multiple examples (shots) to follow.\n\n"

# --- Shot 1 (Example 1) ---
link_system_prompt += "### Shot 1 Input:\n"
link_system_prompt += """
[
    "https://example.com/login",
    "https://example.com/about-us",
    "https://example.com/products/all",
    "https://example.com/jobs"
]
"""
link_system_prompt += "### Shot 1 Output:\n"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://example.com/about-us"},
        {"type": "careers page", "url": "https://example.com/jobs"}
    ]
}
"""

# --- Shot 2 (Example 2) ---
link_system_prompt += "### Shot 2 Input:\n"
link_system_prompt += """
[
    "/contact",
    "https://example-site.org/support",
    "https://example-site.org/investor-relations",
    "https://example-site.org/team"
]
"""
link_system_prompt += "### Shot 2 Output:\n"
link_system_prompt += """
{
    "links": [
        {"type": "investor relations page", "url": "https://example-site.org/investor-relations"},
        {"type": "team page", "url": "https://example-site.org/team"}
    ]
}
"""

# --- End of Shots (Actual Request Template) ---
link_system_prompt += "### Actual Request Input:\n"
link_system_prompt += """
[
    # Insert the actual list of scraped links here
]
"""
link_system_prompt += "### Actual Request Output:\n"
link_system_prompt += """
{
    "links": [
        # Model should complete this JSON based on the input above
    ]
}
"""

In [14]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [15]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/#wp--skip-link--target
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com/
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-

In [16]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [17]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['https://huggingface.co/',
 'https://huggingface.co/models',
 'https://huggingface.co/datasets',
 'https://huggingface.co/spaces',
 'https://huggingface.co/docs',
 'https://huggingface.co/enterprise',
 'https://huggingface.co/pricing',
 'https://huggingface.co/login',
 'https://huggingface.co/join',
 'https://huggingface.co/spaces',
 'https://huggingface.co/models',
 'https://huggingface.co/tencent/HunyuanImage-3.0',
 'https://huggingface.co/deepseek-ai/DeepSeek-V3.2-Exp',
 'https://huggingface.co/tencent/Hunyuan3D-Part',
 'https://huggingface.co/zai-org/GLM-4.6',
 'https://huggingface.co/ServiceNow-AI/Apriel-1.5-15b-Thinker',
 'https://huggingface.co/models',
 'https://huggingface.co/spaces/Wan-AI/Wan2.2-Animate',
 'https://huggingface.co/spaces/enzostvs/deepsite',
 'https://huggingface.co/spaces/zerogpu-aoti/wan2-2-fp8da-aoti-faster',
 'https://huggingface.co/spaces/akhaliq/HunyuanImage-3.0',
 'https://huggingface.co/spaces/multimodalart/ai-toolkit',
 'https://huggingface.co/spaces'

In [18]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [19]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'about page', 'url': 'https://huggingface.co/docs'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'}, {'type': 'support forum', 'url': 'https://discuss.huggingface.co/'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www

In [20]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'team page', 'url': 'https://huggingface.co/team'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\n\nWebpage Contents (Structured):\n# The AI community building the future.\n\nThe platform where the machine learning community collaborates on models, datasets, and applications.\n\n\n## Models\n\n\n## Spaces\n\nWan2.2 Animate\n\nGenerate any application by Vibe Coding\n\ngenerate a video from an image with a text prompt\n\nGenerate images from text prompts\n\nTrain FLUX, Qwen and Wan LoRAs with Ostris Ai Toolkit\n\n\n## Datasets\n\n\n## The Home of Machine Learning\n\nCreate, discover and collaborate on ML better.\n\n\n### The collaboration platform\n\nHost and collaborate on unlimited public models, datasets and applications.\n\n\n### Move faster\n\nWith the HF Open source stack.\n\n\n### Explore all modaliti

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'brand page', 'url': 'https://huggingface.co/brand'}, {'type': 'team page', 'url': 'https://huggingface.co/team'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co/'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/hugging

```markdown
# Hugging Face Brochure

## Welcome to Hugging Face
**The AI community building the future.**  
Join the revolution where the machine learning community collaborates on models, datasets, and applications. 

---

## About Us
At Hugging Face, we are proud to be the home of machine learning, enabling you to create, discover, and collaborate on ML better than ever before. With our open-source stack, you can move faster than ever and explore all modalities including text, image, video, audio, and even 3D.

### Our Collaboration Platform
- Host and collaborate on unlimited public models, datasets, and applications. 
- Build your portfolio and share your work with the world to enhance your ML profile.

## Services We Offer
- **Compute Solutions**: Deploy on optimized Inference Endpoints or update your Spaces applications to include GPU support.
  - Starting at $0.60/hour for GPU.
  
- **Team & Enterprise Solutions**: Empower your team with the most advanced platform in AI that includes enterprise-grade security, access controls, and dedicated support.
  - Starting at $20/user/month.

---

## Customer Base
Join over **50,000 organizations** using Hugging Face to leverage state-of-the-art AI models, datasets, and tools. Our community-driven approach means you’re partnering with innovative minds pushing the boundaries of what’s possible in machine learning.

---

## Open Source Community
Our foundation is built on open-source principles. We're committed to providing state-of-the-art AI tooling, including:
- AI models optimized for PyTorch.
- Fast tokenizers and neural network weight storage solutions.
- A Python client for easy interaction with the Hugging Face Hub. 
- Tools for training and serving language models with optimized toolkits.

---

## Company Culture
At Hugging Face, we foster a culture of collaboration, creativity, and open communication. Our team thrives in an environment that encourages exploration and innovation in the realm of AI. We believe in empowering our employees and fostering an inclusive workplace where everyone’s ideas are valued.

---

## Careers at Hugging Face
We are always on the lookout for talented individuals who are passionate about artificial intelligence and machine learning. If you’re interested in joining a community that values creativity and collaboration, we invite you to explore our career opportunities.

---

**Join us at Hugging Face and be a part of the AI future!**

### Contact Us
For more information, visit our website [HuggingFace.com](https://huggingface.co) and start your machine learning journey with us today!
```


In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co/'}, {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'}, {'type': 'status page', 'url': 'https://status.huggingface.co/'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}



# Hugging Face Brochure

## **Who We Are**
Hugging Face is the AI community building the future. We are dedicated to democratizing machine learning through collaboration, innovation, and open-source contributions. Our platform serves as a hub where machine learning enthusiasts, researchers, and enterprises come together to develop, share, and collaborate on cutting-edge models, datasets, and applications.

## **What We Offer**
- **Models**: Collaborate and host unlimited public models.
- **Spaces**: A collaborative space to generate applications easily, create videos from images with text prompts, and much more.
- **Datasets**: Access and share datasets for any ML tasks, enabling researchers to streamline their projects.

### **Key Features**
- **Collaboration Platform**: Unlimited hosting for models, datasets, and applications.
- **Explore Modalities**: Discover capabilities in text, image, video, audio, and even 3D.
- **Accelerated ML**: Paid compute solutions starting at $0.60/hour for GPU, and enterprise support starting at $20/user/month.

## **Community Impact**
More than **50,000 organizations** rely on Hugging Face for their machine learning needs. We are building a vibrant community focused on open-source tooling that includes:
- State-of-the-art AI models for PyTorch
- Fast tokenizers for research and production
- Parameter-efficient fine-tuning for large language models
- Tools to serve language models with optimized toolkits

## **Join Our Mission**
At Hugging Face, we're on a mission to democratize machine learning, one commit at a time. Our culture promotes innovation, collaboration, and inclusivity. We value creativity and are always looking for like-minded individuals to join our team. If you are passionate about AI and want to make a difference in the industry, explore career opportunities with us.

![Join Us](https://huggingface.co/join)

## **Careers at Hugging Face**
We are constantly looking for talent across various fields to help us push the boundaries of what's possible in AI. Whether you're a machine learning engineer, a researcher, or a marketing guru, there's a place for you in our community. [Explore our job openings](https://huggingface.co/careers).

## **Get Involved**
Join our community and be part of the future of AI:
- Collaborate on projects
- Contribute to open-source models
- Build your ML portfolio and showcase your work

For press inquiries and more information, please contact our team directly at [contact@huggingface.co](mailto:contact@huggingface.co).

---

**Hugging Face**: Where the future of AI is created collaboratively.



In [27]:
system_prompt_arabic = "Translate this Brochure to arabic"
user_prompt = "here is the Brochure that you should translate to arabic"
user_prompt += stream_brochure("HuggingFace", "https://huggingface.co")
def translate_arabic(user_prompt):
    messages = [
        {"role":"system","content":system_prompt_arabic},
        {"role":"user","content":user_prompt}
    ]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    return response.choices[0].message.content

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'about page', 'url': 'https://huggingface.co/about'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'team page', 'url': 'https://huggingface.co/team'}, {'type': 'learn page', 'url': 'https://huggingface.co/learn'}]}



# Hugging Face Brochure

Welcome to Hugging Face, the AI community building the future! Our platform is where the machine learning community comes together to collaborate on models, datasets, and applications. 

## About Us

At Hugging Face, we believe in the power of collaboration to create groundbreaking solutions in machine learning (ML). Our tools allow users to create, discover, and collaborate more effectively than ever before. With over **50,000 organizations** leveraging our technology, we are proud to serve as the home for machine learning innovation.

## What We Offer

### Models
Explore and utilize state-of-the-art AI models designed for various applications, including natural language processing, image generation, and more.

### Datasets
Access a comprehensive repository of datasets tailored for any machine learning task, promoting an open and collaborative research environment.

### Spaces
Create and deploy interactive applications using our innovative Vibe Coding platform. Generate videos from text prompts and images seamlessly.

## Collaboration Platform

- **Unlimited Public Models and Datasets**: Host and collaborate on public resources to help advance the ML community.
- **Open Source Stack**: Move faster with tools that benefit everyone, from researchers to developers.
- **Explore All Modalities**: Work with text, images, videos, audio, and 3D data.

## Accelerate Your ML

We provide compute options and enterprise solutions to meet your organization's needs:

- **Compute**: Starting at **$0.60/hour** for GPU, easily deploy optimized inference endpoints and applications.
- **Team & Enterprise Solutions**: Advanced platform access beginning at **$20/user/month**, featuring enterprise-grade security, access controls, and dedicated support.

## Company Culture

At Hugging Face, our community-driven culture fosters innovation, creativity, and collaboration. We prioritize open communication and encourage our team to share their knowledge and experiences. Join us to be part of a supportive environment that values growth and passionate contributions.

## Careers at Hugging Face

We are always on the lookout for talented individuals who are eager to shape the future of AI. If you're passionate about machine learning and want to make a global impact, explore our careers page for exciting opportunities to join our diverse and inclusive team.

---

**Join Us Today!**

Let's build the future of AI together. Become a part of the Hugging Face community!

[Visit our website](https://huggingface.co) for more information.



TypeError: can only concatenate str (not "NoneType") to str